# RAG-IDEArq — Evaluación con Langfuse + Ollama Local

- **Dataset**: `RAG-IDEArq-eval-v3-SIMPLE` (Langfuse)
- **Prompts**: cargados desde Langfuse (`prompt_zero_shot`, `prompt_one_shot`, `prompt_few_shot`)
- **LLMs**: Phi-4-mini, Qwen3-4B-Instruct-2507, Llama-3.2-3B-Instruct
- **Embeddings**: all-MiniLM-L6-v2, gte-multilingual-base, e5-large-instruct
- **Judge RAGAS**: qwen2.5:14b (Ollama local)
- **k=10, 10 docs al contexto, sin reranking**
- **Langfuse**: traces con input/output/retrieval_breakdown/scores

In [ ]:
# Cell 1: Setup
from __future__ import annotations
import os, sys, time, json, itertools, warnings
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Optional

warnings.filterwarnings('ignore')

try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(Path.cwd() / ".env", override=True)

# Langfuse
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler

langfuse_client = Langfuse(
    public_key=os.getenv('LANGFUSE_PUBLIC_KEY'),
    secret_key=os.getenv('LANGFUSE_SECRET_KEY'),
    host=os.getenv('LANGFUSE_BASE_URL', 'http://localhost:4000'),
)
langfuse_handler = CallbackHandler()
print(f"Langfuse: {langfuse_client is not None} | host={os.getenv('LANGFUSE_BASE_URL')}")

# Config
from src.config import (
    EMBEDDINGS, LLMS, LLM_TEMPERATURES, RERANKING_CONFIG,
    collection_name, RESULTS_DIR,
)

import weaviate
from langchain_weaviate import WeaviateVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM, ChatOllama
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import ChatPromptTemplate
from sentence_transformers import SentenceTransformer
import torch
import pandas as pd
import requests
from requests.auth import HTTPBasicAuth

print(f"Embeddings: {list(EMBEDDINGS.keys())}")
print(f"LLMs: {list(LLMS.keys())}")
print(f"Temperatures: {LLM_TEMPERATURES}")
print(f"Reranking: {RERANKING_CONFIG['enabled']}")
print(f"Langfuse handler: {langfuse_handler is not None}")

In [ ]:
# Cell 2: Load embedding models
class E5InstructEmbeddings(Embeddings):
    def __init__(self, model_name="intfloat/multilingual-e5-large-instruct", device=None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.model = SentenceTransformer(model_name, device=self.device)
    def embed_documents(self, texts):
        prefixed = [f"passage: {t}" for t in texts]
        return self.model.encode(prefixed, device=self.device).tolist()
    def embed_query(self, text):
        prefixed = f"query: {text}"
        return self.model.encode([prefixed], device=self.device)[0].tolist()

print("Loading embedding models...")
embedding_models = {}
for emb_key, emb_cfg in EMBEDDINGS.items():
    try:
        if emb_cfg.get("model_class") == "E5InstructEmbeddings":
            emb = E5InstructEmbeddings(model_name=emb_cfg["model_name"])
            embedding_models[emb_key] = emb
            print(f"  [OK] {emb_key} on {emb.device} (E5 custom)")
        else:
            model_kwargs = {"device": "cuda"}
            if emb_cfg.get("trust_remote_code", False):
                model_kwargs["trust_remote_code"] = True
            emb = HuggingFaceEmbeddings(
                model_name=emb_cfg["model_name"],
                model_kwargs=model_kwargs,
                encode_kwargs={"device": "cuda"},
            )
            embedding_models[emb_key] = emb
            print(f"  [OK] {emb_key} on CUDA")
    except Exception as e:
        model_kwargs = {"device": "cpu"}
        if emb_cfg.get("trust_remote_code", False):
            model_kwargs["trust_remote_code"] = True
        emb = HuggingFaceEmbeddings(
            model_name=emb_cfg["model_name"],
            model_kwargs=model_kwargs,
            encode_kwargs={"device": "cpu"},
        )
        embedding_models[emb_key] = emb
        print(f"  [CPU] {emb_key}")

In [ ]:
# Cell 3: Load LLM models
print("Loading LLM models...")
llm_models = {}
for llm_name, llm_model in LLMS.items():
    for temp in LLM_TEMPERATURES:
        key = f"{llm_name}_t{temp}"
        llm_models[key] = OllamaLLM(model=llm_model, temperature=temp)
        print(f"  [OK] {key}")

In [ ]:
# Cell 4: Load dataset from Langfuse
DATASET_NAME = "RAG-IDEArq-eval-v3-SIMPLE"
dataset = langfuse_client.get_dataset(name=DATASET_NAME)
print(f"Dataset: {dataset.name}")
print(f"Items: {len(dataset.items)}")
for i, item in enumerate(dataset.items[:3]):
    print(f"\nQ{i+1}: {str(item.input)[:80]}...")
    print(f"  Expected: {str(item.expected_output)[:80]}...")

In [ ]:
# Cell 5: Load prompts from Langfuse
PROMPTS = {}
PROMPT_NAMES = {
    "zero_shot": "prompt_zero_shot",
    "one_shot": "prompt_one_shot",
    "few_shot": "prompt_few_shot",
}
for key, name in PROMPT_NAMES.items():
    try:
        PROMPTS[key] = langfuse_client.get_prompt(name=name, type="chat")
        print(f"  [OK] {key} loaded from Langfuse")
    except Exception:
        try:
            PROMPTS[key] = langfuse_client.get_prompt(name=name, type="text")
            print(f"  [OK] {key} loaded from Langfuse (text)")
        except Exception as e:
            print(f"  [ERROR] {key}: {e}")
print(f"\nPrompts loaded: {list(PROMPTS.keys())}")

In [ ]:
# Cell 6: RAGAS setup with Ollama local judge
from ragas.llms.base import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ContextPrecision, ContextRecall, AnswerCorrectness
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from ragas import evaluate as ragas_evaluate
from ragas.dataset_schema import SingleTurnSample
from ragas import EvaluationDataset

class JSONCleaningLLM(LangchainLLMWrapper):
    def __call__(self, prompt, **kwargs):
        response = super().__call__(prompt, **kwargs)
        if isinstance(response, str):
            if response.startswith("```"):
                response = response.split("\n", 1)[1]
                if response.endswith("```"):
                    response = response.rsplit("\n", 1)[0]
            try:
                start = response.find("{")
                end = response.rfind("}")
                if start >= 0 and end > start:
                    response = response[start:end+1]
            except Exception:
                pass
        return response

RAGAS_JUDGE_MODEL = os.getenv("RAGAS_JUDGE_MODEL", "qwen2.5:14b")
print(f"RAGAS Judge: {RAGAS_JUDGE_MODEL} (Ollama local)")

raw_judge = ChatOllama(model=RAGAS_JUDGE_MODEL, temperature=0.1, format="json", num_ctx=32768)
ragas_llm = JSONCleaningLLM(raw_judge)
ragas_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                          model_kwargs={"device": "cpu"},
                          encode_kwargs={"device": "cpu"})
)
ragas_metrics = [
    Faithfulness(llm=ragas_llm),
    ContextPrecision(llm=ragas_llm),
    ContextRecall(llm=ragas_llm),
    AnswerCorrectness(llm=ragas_llm, embeddings=ragas_emb),
]
ragas_run_config = RunConfig(max_workers=1, max_retries=10, max_wait=120, timeout=900)
print("RAGAS metrics configured")

In [ ]:
# Cell 7: Build RAG chain and run_rag function
w_client = weaviate.connect_to_local(host='localhost', port=8080, grpc_port=50051)
print(f"Weaviate: {w_client.is_ready()}")
print(f"Collections: {list(w_client.collections.list_all().keys())}")

def build_rag_chain(emb_key, prompt_key):
    coll_name = collection_name(emb_key)
    if not w_client.collections.exists(coll_name):
        raise ValueError(f"Collection '{coll_name}' does not exist.")
    vs = WeaviateVectorStore(
        client=w_client,
        index_name=coll_name,
        text_key="content",
        embedding=embedding_models[emb_key],
        attributes=["filename", "source", "chunk_index", "year", "language", "doi",
                    "authors", "periodo", "region", "doc_type", "yacimiento_nombre",
                    "tipologia_crono", "unidad_territorial"],
    )
    retriever = vs.as_retriever(search_type="similarity", search_kwargs={"k": 10})
    prompt_template = PROMPTS[prompt_key]
    return retriever, prompt_template

def run_rag(retriever, prompt_template, llm, question, trace_id=None):
    docs = retriever.invoke(question)
    # Log retrieval breakdown with doc metadata
    if trace_id:
        doc_info = [
            {"filename": d.metadata.get("filename", "N/A"),
             "doc_type": d.metadata.get("doc_type", "N/A"),
             "yacimiento_nombre": d.metadata.get("yacimiento_nombre", "N/A"),
             "tipologia_crono": d.metadata.get("tipologia_crono", "N/A"),
             "unidad_territorial": d.metadata.get("unidad_territorial", "N/A"),
             "year": d.metadata.get("year", "N/A"),
             "language": d.metadata.get("language", "N/A")}
            for d in docs[:10]
        ]
        try:
            langfuse_client.create_event(
                trace_context={"trace_id": trace_id},
                name="retrieval_breakdown",
                input={"question": question},
                output={"weaviate_docs": len(docs), "documents": doc_info},
                metadata={"k": 10, "rerank": False},
            )
        except Exception:
            pass
    # 10 docs to context
    context = "\n\n".join([d.page_content for d in docs[:10]])
    if hasattr(prompt_template, 'format'):
        prompt_text = prompt_template.format(context=context, question=question)
    else:
        prompt_text = prompt_template.compile(context=context, question=question)
    answer = llm.invoke(prompt_text, config={"callbacks": [langfuse_handler]})
    filenames = [d.metadata.get("filename", "N/A") for d in docs[:10]]
    return answer, docs, filenames

print("RAG chain functions defined")

In [ ]:
# Cell 8: Smoke test
print("="*60)
print("SMOKE TEST: GTE + Llama-3.2-3B + k=10 + 10 docs context")
print("="*60)

SMOKE_EMB = "gte-multilingual-base"
SMOKE_LLM = "Llama-3.2-3B-Instruct"
SMOKE_PROMPT = "zero_shot"
SMOKE_TEMP = 0.3

item = dataset.items[0]
question = item.input if isinstance(item.input, str) else item.input.get("question", str(item.input))
ground_truth = item.expected_output if isinstance(item.expected_output, str) else str(item.expected_output)
print(f"\nQuestion: {str(question)[:80]}...")

with langfuse_client.start_as_current_observation(name="smoke_test", as_type="span") as span:
    trace_id = langfuse_client.get_current_trace_id()
    print(f"Trace ID: {trace_id}")
    span.update(input={"question": question})

    retriever, prompt_template = build_rag_chain(SMOKE_EMB, SMOKE_PROMPT)
    llm = llm_models[f"{SMOKE_LLM}_t{SMOKE_TEMP}"]

    answer, docs, filenames = run_rag(retriever, prompt_template, llm, question, trace_id)
    print(f"\n✅ Answer ({len(answer)} chars):")
    print(answer)
    print(f"\n📄 Documents consulted:")
    for i, fn in enumerate(filenames):
        print(f"  {i+1}. {fn}")

    span.update(output={"answer": answer})

    # RAGAS
    print("\n[4/5] RAGAS...")
    sample = SingleTurnSample(
        user_input=question,
        response=answer,
        reference=ground_truth,
        retrieved_contexts=[d.page_content[:800] for d in docs[:5]],
    )
    eval_ds = EvaluationDataset(samples=[sample])
    try:
        ragas_result = ragas_evaluate(dataset=eval_ds, metrics=ragas_metrics, llm=ragas_llm, embeddings=ragas_emb, run_config=ragas_run_config)
        print(f"✅ RAGAS scores:")
        for metric, value in ragas_result.scores[0].items():
            print(f"   {metric}: {value:.4f}")
            try:
                langfuse_client.create_score(trace_id=trace_id, name=f"ragas_{metric}", value=float(value), data_type="NUMERIC")
                print(f"   → Scored in Langfuse ✅")
            except Exception:
                pass
    except Exception as e:
        print(f"⚠️ RAGAS error: {e}")

langfuse_client.flush()
time.sleep(10)
print("\n✅ SMOKE TEST COMPLETADO")
print(f"Check Langfuse: http://localhost:4000")
print(f"Trace ID: {trace_id}")

In [ ]:
# Cell 9: Run full evaluation grid
all_results = []

combos = list(itertools.product(
    EMBEDDINGS.keys(),
    LLMS.keys(),
    ["zero_shot", "one_shot", "few_shot"],
    LLM_TEMPERATURES,
))

print(f"\nTotal combos: {len(combos)}")
print(f"Questions per combo: {len(dataset.items)}")
print(f"Total evaluations: {len(combos) * len(dataset.items)}\n")

for emb_key, llm_name, prompt_key, temperature in combos:
    llm_key = f"{llm_name}_t{temperature}"
    combo_label = f"{emb_key}|{llm_name}|{prompt_key}|t{temperature}"
    print(f"\n{'='*60}")
    print(f"Combo: {combo_label}")
    print(f"{'='*60}")

    try:
        retriever, prompt_template = build_rag_chain(emb_key, prompt_key)
        llm = llm_models[llm_key]
    except ValueError as e:
        print(f"  SKIP: {e}")
        continue

    ragas_samples = []
    combo_trace_ids = []

    for idx, item in enumerate(dataset.items):
        question = item.input if isinstance(item.input, str) else item.input.get("question", str(item.input))
        ground_truth = item.expected_output if isinstance(item.expected_output, str) else str(item.expected_output)

        print(f"  Q{idx+1}/{len(dataset.items)}: {str(question)[:60]}...")

        with langfuse_client.start_as_current_observation(name="rag_evaluation", as_type="span") as span:
            trace_id = langfuse_client.get_current_trace_id()
            combo_trace_ids.append(trace_id)
            span.update(input={"question": question, "combo": combo_label})

            t0 = time.time()
            try:
                answer, docs, filenames = run_rag(retriever, prompt_template, llm, question, trace_id)
                latency = time.time() - t0
                span.update(output={"answer": answer})

                contexts = [d.page_content[:800] for d in docs[:5]]
                sample = SingleTurnSample(
                    user_input=question,
                    response=answer,
                    reference=ground_truth,
                    retrieved_contexts=contexts,
                )
                ragas_samples.append(sample)

                all_results.append({
                    "combo": combo_label,
                    "embedding": emb_key,
                    "llm": llm_name,
                    "prompt": prompt_key,
                    "temperature": temperature,
                    "use_rerank": False,
                    "question": question,
                    "answer": answer,
                    "ground_truth": ground_truth[:500],
                    "n_docs_retrieved": len(docs),
                    "filenames_consulted": ";".join(filenames),
                    "latency_s": latency,
                })
                print(f"    OK ({latency:.1f}s, {len(docs)} docs)")
            except Exception as e:
                print(f"    ERROR: {e}")
                all_results.append({
                    "combo": combo_label, "embedding": emb_key, "llm": llm_name,
                    "prompt": prompt_key, "temperature": temperature, "use_rerank": False,
                    "question": question, "answer": f"Error: {e}",
                    "ground_truth": ground_truth[:500], "n_docs_retrieved": 0,
                    "filenames_consulted": "", "latency_s": 0,
                })

    # RAGAS per combo
    if ragas_samples:
        print(f"\n  Running RAGAS ({len(ragas_samples)} samples)...")
        eval_ds = EvaluationDataset(samples=ragas_samples)
        for attempt in range(5):
            try:
                ragas_result = ragas_evaluate(dataset=eval_ds, metrics=ragas_metrics, llm=ragas_llm, embeddings=ragas_emb, run_config=ragas_run_config)
                break
            except Exception as e:
                if "429" in str(e) or "rate_limit" in str(e).lower():
                    wait = 30 * (2 ** attempt)
                    print(f"  [Rate limit] Waiting {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"  [RAGAS Error] {e}")
                    ragas_result = None
                    break

        if ragas_result:
            print(f"  RAGAS results:")
            for metric_name, metric_value in ragas_result.scores[0].items():
                print(f"    {metric_name}: {metric_value:.4f}")
                # Score each trace
                for tid in combo_trace_ids:
                    try:
                        langfuse_client.create_score(trace_id=tid, name=f"ragas_{metric_name}", value=float(metric_value), data_type="NUMERIC")
                    except Exception:
                        pass

    langfuse_client.flush()
    print(f"\n  Waiting 60s before next combo...")
    time.sleep(60)

In [ ]:
# Cell 10: Save results and report
if not all_results:
    print("No results to report.")
else:
    df = pd.DataFrame(all_results)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = Path(RESULTS_DIR) / f"eval_v3_baseline_{timestamp}.csv"
    Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Results saved to: {output_path}")
    print(f"Total rows: {len(df)}")
    print(f"Columns: {list(df.columns)}")

    print("\n" + "="*60)
    print("✅ EVALUACIÓN COMPLETADA")
    print("="*60)
    print(f"\nCSV: {output_path}")
    print(f"Langfuse: http://localhost:4000")
    print(f"Total evaluaciones: {len(df)}")

    w_client.close()